In [1]:
import requests
import tarfile
import io
import os

# --- CONFIG ---
VERSION = "15.24.1"
URL = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{VERSION}.tgz"

DEST_DIR = r"D:\lol draft analyzer\part 2\data champions\collect"
TARGET_PREFIX = f"{VERSION}/data/fr_FR/champion/"

# --- CREATE DEST DIR IF NOT EXISTS ---
os.makedirs(DEST_DIR, exist_ok=True)

# --- DOWNLOAD TGZ INTO MEMORY ---
print("Downloading dragontail...")
response = requests.get(URL, stream=True)
response.raise_for_status()

file_like_object = io.BytesIO(response.content)

# --- OPEN TGZ ---
with tarfile.open(fileobj=file_like_object, mode="r:gz") as tar:
    members = tar.getmembers()

    champion_jsons = [
        m for m in members
        if m.name.startswith(TARGET_PREFIX) and m.name.endswith(".json")
    ]

    print(f"Found {len(champion_jsons)} champion JSON files")

    for member in champion_jsons:
        # Extract file object
        extracted_file = tar.extractfile(member)
        if extracted_file is None:
            continue

        filename = os.path.basename(member.name)
        output_path = os.path.join(DEST_DIR, filename)

        with open(output_path, "wb") as f:
            f.write(extracted_file.read())

        print(f"Saved: {filename}")

print("✅ Done.")


Found 172 champion JSON files
Saved: Aatrox.json
Saved: Ahri.json
Saved: Akali.json
Saved: Akshan.json
Saved: Alistar.json
Saved: Ambessa.json
Saved: Amumu.json
Saved: Anivia.json
Saved: Annie.json
Saved: Aphelios.json
Saved: Ashe.json
Saved: AurelionSol.json
Saved: Aurora.json
Saved: Azir.json
Saved: Bard.json
Saved: Belveth.json
Saved: Blitzcrank.json
Saved: Brand.json
Saved: Braum.json
Saved: Briar.json
Saved: Caitlyn.json
Saved: Camille.json
Saved: Cassiopeia.json
Saved: Chogath.json
Saved: Corki.json
Saved: Darius.json
Saved: Diana.json
Saved: DrMundo.json
Saved: Draven.json
Saved: Ekko.json
Saved: Elise.json
Saved: Evelynn.json
Saved: Ezreal.json
Saved: Fiddlesticks.json
Saved: Fiora.json
Saved: Fizz.json
Saved: Galio.json
Saved: Gangplank.json
Saved: Garen.json
Saved: Gnar.json
Saved: Gragas.json
Saved: Graves.json
Saved: Gwen.json
Saved: Hecarim.json
Saved: Heimerdinger.json
Saved: Hwei.json
Saved: Illaoi.json
Saved: Irelia.json
Saved: Ivern.json
Saved: Janna.json
Saved: Jarvan

In [3]:
print("\n=== EXTRACTION PHASE ===")
print(f"Destination directory exists: {os.path.exists(DEST_DIR)}")
print(f"Destination directory writable: {os.access(DEST_DIR, os.W_OK)}")
print("========================\n")

for idx, m in enumerate(json_files, start=1):
    print(f"\n[{idx}/{len(json_files)}] Processing member:")
    print(" - Archive path:", m.name)
    print(" - Size in archive:", m.size)

    # --- extract file object ---
    try:
        extracted = tar.extractfile(m)
        if extracted is None:
            print(" ⚠ extractfile() returned None")
            continue
    except Exception as e:
        print(" ❌ Exception during extractfile():", repr(e))
        continue

    # --- read bytes ---
    try:
        content = extracted.read()
        print(" - Bytes read:", len(content))
    except Exception as e:
        print(" ❌ Exception while reading extracted file:", repr(e))
        continue

    if len(content) == 0:
        print(" ⚠ File is empty, skipping write")
        continue

    filename = os.path.basename(m.name)
    output_path = os.path.join(DEST_DIR, filename)

    print(" - Output path:", output_path)

    # --- write to disk ---
    try:
        with open(output_path, "wb") as f:
            f.write(content)
        print(" ✔ Write finished")
    except PermissionError as e:
        print(" ❌ PermissionError while writing:", repr(e))
        continue
    except OSError as e:
        print(" ❌ OSError while writing:", repr(e))
        continue
    except Exception as e:
        print(" ❌ Unknown error while writing:", repr(e))
        continue

    # --- verify write ---
    if os.path.exists(output_path):
        size = os.path.getsize(output_path)
        print(f" ✔ File exists on disk ({size} bytes)")
    else:
        print(" ❌ File NOT found after write")

print("\n=== EXTRACTION COMPLETE ===")



=== EXTRACTION PHASE ===
Destination directory exists: True
Destination directory writable: True


[1/172] Processing member:
 - Archive path: 15.24.1/data/fr_FR/champion/Aatrox.json
 - Size in archive: 9704
 ❌ Exception during extractfile(): OSError('TarFile is closed')

[2/172] Processing member:
 - Archive path: 15.24.1/data/fr_FR/champion/Ahri.json
 - Size in archive: 9455
 ❌ Exception during extractfile(): OSError('TarFile is closed')

[3/172] Processing member:
 - Archive path: 15.24.1/data/fr_FR/champion/Akali.json
 - Size in archive: 9997
 ❌ Exception during extractfile(): OSError('TarFile is closed')

[4/172] Processing member:
 - Archive path: 15.24.1/data/fr_FR/champion/Akshan.json
 - Size in archive: 9415
 ❌ Exception during extractfile(): OSError('TarFile is closed')

[5/172] Processing member:
 - Archive path: 15.24.1/data/fr_FR/champion/Alistar.json
 - Size in archive: 8868
 ❌ Exception during extractfile(): OSError('TarFile is closed')

[6/172] Processing member:
 - Ar

In [4]:
import requests
import tarfile
import io
import os
import json
import csv

# ================== CONFIG ==================
VERSIONS = [
    "15.24.1",
    # "15.23.1",
    # ajoute d'autres versions ici
]

LANG = "fr_FR"

DEST_DIR = r"D:\lol draft analyzer\part 2\data champions\collect"
CSV_OUTPUT = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"
# ============================================

os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)

# --- CSV SETUP ---
csv_exists = os.path.exists(CSV_OUTPUT)
csv_file = open(CSV_OUTPUT, "a", newline="", encoding="utf-8")
csv_writer = csv.writer(csv_file)

if not csv_exists:
    csv_writer.writerow([
        "patch",
        "champion_id",
        "champion_name",
        "title"
    ])

# ================== MAIN LOOP ==================
for version in VERSIONS:
    print(f"\n=== PATCH {version} ===")

    url = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{version}.tgz"
    print("Downloading:", url)

    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print("❌ Download failed:", e)
        continue

    print("✔ Download OK - bytes:", len(response.content))
    tar_bytes = io.BytesIO(response.content)

    # -------- OPEN ARCHIVE --------
    try:
        with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
            prefix = f"{version}/data/{LANG}/champion/"

            members = [
                m for m in tar.getmembers()
                if m.name.startswith(prefix) and m.name.endswith(".json")
            ]

            print(f"✔ Found {len(members)} champion JSON files")

            # -------- EXTRACTION + CSV --------
            for m in members:
                try:
                    extracted = tar.extractfile(m)
                    if extracted is None:
                        print("⚠ Skipped (not a file):", m.name)
                        continue

                    data = json.load(extracted)
                except Exception as e:
                    print("❌ Error reading JSON:", m.name, e)
                    continue

                champ = next(iter(data["data"].values()))

                # --- WRITE JSON FILE ---
                filename = os.path.basename(m.name)
                output_path = os.path.join(DEST_DIR, filename)

                try:
                    with open(output_path, "w", encoding="utf-8") as f:
                        json.dump(data, f, ensure_ascii=False, indent=2)
                except Exception as e:
                    print("❌ Write JSON failed:", output_path, e)
                    continue

                # --- WRITE CSV LINE ---
                csv_writer.writerow([
                    version,
                    champ.get("id"),
                    champ.get("name"),
                    champ.get("title")
                ])

        print(f"✅ Patch {version} processed successfully")

    except Exception as e:
        print("❌ Archive processing failed:", e)

csv_file.close()
print("\n🎉 ALL DONE")



=== PATCH 15.24.1 ===
Downloading: https://ddragon.leagueoflegends.com/cdn/dragontail-15.24.1.tgz
✔ Download OK - bytes: 2214693408
✔ Found 172 champion JSON files
✅ Patch 15.24.1 processed successfully

🎉 ALL DONE


In [5]:
import pandas as pd

csv_path = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"

df = pd.read_csv(csv_path)

print(df.head())


     patch champion_id champion_name                  title
0  15.24.1      Aatrox        Aatrox        Épée des Darkin
1  15.24.1        Ahri          Ahri  Renarde à neuf queues
2  15.24.1       Akali         Akali       Assassin rebelle
3  15.24.1      Akshan        Akshan     Sentinelle rebelle
4  15.24.1     Alistar       Alistar              Minotaure


In [9]:
import requests
import tarfile
import io
import os
import json
import csv

# ================== CONFIG ==================
VERSIONS = [
    "15.24.1",
]

LANG = "fr_FR"

DEST_DIR = r"D:\lol draft analyzer\part 2\data champions\collect"
CSV_OUTPUT = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"
# ============================================

os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)

# ================== CSV SETUP ==================
csv_exists = os.path.exists(CSV_OUTPUT)
csv_file = open(CSV_OUTPUT, "a", newline="", encoding="utf-8")
writer = csv.writer(csv_file)

HEADERS = [
    "patch",
    "id",
    "key",
    "name",
    "title",
    "partype",
    "tags",
    "attack",
    "defense",
    "magic",
    "difficulty",
    "hp",
    "hpperlevel",
    "mp",
    "mpperlevel",
    "armor",
    "armorperlevel",
    "spellblock",
    "spellblockperlevel",
    "attackdamage",
    "attackdamageperlevel",
    "attackspeed",
    "attackspeedperlevel",
    "attackrange",
    "movespeed",
    "hpregen",
    "hpregenperlevel",
    "mpregen",
    "mpregenperlevel",
    "crit",
    "critperlevel",
    "passive_name",
    "passive_description",
    "spell_q",
    "spell_w",
    "spell_e",
    "spell_r",
    "skins_count"
]

if not csv_exists:
    writer.writerow(HEADERS)

# ================== MAIN LOOP ==================
for version in VERSIONS:
    print(f"\n=== PATCH {version} ===")

    url = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{version}.tgz"
    print("Downloading:", url)

    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print("❌ Download failed:", e)
        continue

    tar_bytes = io.BytesIO(response.content)

    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        prefix = f"{version}/data/{LANG}/champion/"

        members = [
            m for m in tar.getmembers()
            if m.name.startswith(prefix) and m.name.endswith(".json")
        ]

        print(f"✔ Found {len(members)} champions")

        for m in members:
            try:
                extracted = tar.extractfile(m)
                if extracted is None:
                    continue

                raw = json.load(extracted)
                champ = next(iter(raw["data"].values()))
            except Exception as e:
                print("❌ JSON error:", m.name, e)
                continue

            # --- OPTIONAL: save raw JSON ---
            out_json = os.path.join(DEST_DIR, os.path.basename(m.name))
            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(raw, f, ensure_ascii=False, indent=2)

            info = champ.get("info", {})
            stats = champ.get("stats", {})
            passive = champ.get("passive", {})
            spells = champ.get("spells", [])

            row = [
                version,
                champ.get("id"),
                champ.get("key"),
                champ.get("name"),
                champ.get("title"),
                champ.get("partype"),
                ",".join(champ.get("tags", [])),
                info.get("attack"),
                info.get("defense"),
                info.get("magic"),
                info.get("difficulty"),
                stats.get("hp"),
                stats.get("hpperlevel"),
                stats.get("mp"),
                stats.get("mpperlevel"),
                stats.get("armor"),
                stats.get("armorperlevel"),
                stats.get("spellblock"),
                stats.get("spellblockperlevel"),
                stats.get("attackdamage"),
                stats.get("attackdamageperlevel"),
                stats.get("attackspeed"),
                stats.get("attackspeedperlevel"),
                stats.get("attackrange"),
                stats.get("movespeed"),
                stats.get("hpregen"),
                stats.get("hpregenperlevel"),
                stats.get("mpregen"),
                stats.get("mpregenperlevel"),
                stats.get("crit"),
                stats.get("critperlevel"),
                passive.get("name"),
                passive.get("description"),
                spells[0]["name"] if len(spells) > 0 else None,
                spells[1]["name"] if len(spells) > 1 else None,
                spells[2]["name"] if len(spells) > 2 else None,
                spells[3]["name"] if len(spells) > 3 else None,
                len(champ.get("skins", [])),
            ]

            writer.writerow(row)

    print(f"✅ Patch {version} done")

csv_file.close()
print("\n🎉 CSV COMPLETED")



=== PATCH 15.24.1 ===
Downloading: https://ddragon.leagueoflegends.com/cdn/dragontail-15.24.1.tgz
✔ Found 172 champions
✅ Patch 15.24.1 done

🎉 CSV COMPLETED


In [10]:
import pandas as pd

csv_path = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"

df = pd.read_csv(csv_path)

print(df.head())

     patch       id  key     name                  title        partype  \
0  15.24.1   Aatrox  266   Aatrox        Épée des Darkin  Puits de sang   
1  15.24.1     Ahri  103     Ahri  Renarde à neuf queues           Mana   
2  15.24.1    Akali   84    Akali       Assassin rebelle        Énergie   
3  15.24.1   Akshan  166   Akshan     Sentinelle rebelle           Mana   
4  15.24.1  Alistar   12  Alistar              Minotaure           Mana   

                tags  attack  defense  magic  ...  mpregenperlevel  crit  \
0            Fighter       8        4      3  ...              0.0     0   
1      Mage,Assassin       3        4      8  ...              0.8     0   
2           Assassin       5        3      8  ...              0.0     0   
3  Marksman,Assassin       0        0      0  ...              0.7     0   
4       Tank,Support       6        9      5  ...              0.8     0   

   critperlevel           passive_name  \
0             0  Posture du massacreur   
1       

In [11]:
print(df.columns)

Index(['patch', 'id', 'key', 'name', 'title', 'partype', 'tags', 'attack',
       'defense', 'magic', 'difficulty', 'hp', 'hpperlevel', 'mp',
       'mpperlevel', 'armor', 'armorperlevel', 'spellblock',
       'spellblockperlevel', 'attackdamage', 'attackdamageperlevel',
       'attackspeed', 'attackspeedperlevel', 'attackrange', 'movespeed',
       'hpregen', 'hpregenperlevel', 'mpregen', 'mpregenperlevel', 'crit',
       'critperlevel', 'passive_name', 'passive_description', 'spell_q',
       'spell_w', 'spell_e', 'spell_r', 'skins_count'],
      dtype='object')


In [15]:
import requests
import tarfile
import io
import os
import json
import csv

# ================== CONFIG ==================
VERSIONS = [
    "15.24.1",
]

LANG = "fr_FR"

DEST_DIR = r"D:\lol draft analyzer\part 2\data champions\collect"
CSV_OUTPUT = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"
# ============================================

os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)

# ================== CSV SETUP ==================
csv_exists = os.path.exists(CSV_OUTPUT)
csv_file = open(CSV_OUTPUT, "a", newline="", encoding="utf-8")
writer = csv.writer(csv_file)

HEADERS = [
    "patch",
    "id",
    "key",
    "name",
    "title",
    "partype",
    "tags",

    # info
    "attack",
    "defense",
    "magic",
    "difficulty",

    # stats
    "hp","hpperlevel","mp","mpperlevel","armor","armorperlevel",
    "spellblock","spellblockperlevel",
    "attackdamage","attackdamageperlevel",
    "attackspeed","attackspeedperlevel",
    "attackrange","movespeed",
    "hpregen","hpregenperlevel",
    "mpregen","mpregenperlevel",
    "crit","critperlevel",

    # passive
    "passive_name",
    "passive_description",

    # SPELL Q
    "spell_q_name",
    "spell_q_maxrank",
    "spell_q_cooldown",
    "spell_q_cost",
    "spell_q_range",
    "spell_q_effects",

    # SPELL W
    "spell_w_name",
    "spell_w_maxrank",
    "spell_w_cooldown",
    "spell_w_cost",
    "spell_w_range",
    "spell_w_effects",

    # SPELL E
    "spell_e_name",
    "spell_e_maxrank",
    "spell_e_cooldown",
    "spell_e_cost",
    "spell_e_range",
    "spell_e_effects",

    # SPELL R
    "spell_r_name",
    "spell_r_maxrank",
    "spell_r_cooldown",
    "spell_r_cost",
    "spell_r_range",
    "spell_r_effects",

    "skins_count"
]

if not csv_exists:
    writer.writerow(HEADERS)

# ================== UTILS ==================
def extract_spell(spells, index):
    if len(spells) <= index:
        return [None]*6

    s = spells[index]
    effects = []

    for e in s.get("effectBurn", []):
        if e and e != "0":
            effects.append(e)

    return [
        s.get("name"),
        s.get("maxrank"),
        s.get("cooldownBurn"),
        s.get("costBurn"),
        s.get("rangeBurn"),
        " | ".join(effects) if effects else None
    ]

# ================== MAIN LOOP ==================
for version in VERSIONS:
    print(f"\n=== PATCH {version} ===")

    url = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{version}.tgz"

    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print("❌ Download failed:", e)
        continue

    tar_bytes = io.BytesIO(response.content)

    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        prefix = f"{version}/data/{LANG}/champion/"

        members = [
            m for m in tar.getmembers()
            if m.name.startswith(prefix) and m.name.endswith(".json")
        ]

        print(f"✔ Found {len(members)} champions")

        for m in members:
            try:
                extracted = tar.extractfile(m)
                raw = json.load(extracted)
                champ = next(iter(raw["data"].values()))
            except Exception:
                continue

            info = champ.get("info", {})
            stats = champ.get("stats", {})
            passive = champ.get("passive", {})
            spells = champ.get("spells", [])

            q = extract_spell(spells, 0)
            w = extract_spell(spells, 1)
            e = extract_spell(spells, 2)
            r = extract_spell(spells, 3)

            row = [
                version,
                champ.get("id"),
                champ.get("key"),
                champ.get("name"),
                champ.get("title"),
                champ.get("partype"),
                ",".join(champ.get("tags", [])),

                info.get("attack"),
                info.get("defense"),
                info.get("magic"),
                info.get("difficulty"),

                stats.get("hp"), stats.get("hpperlevel"),
                stats.get("mp"), stats.get("mpperlevel"),
                stats.get("armor"), stats.get("armorperlevel"),
                stats.get("spellblock"), stats.get("spellblockperlevel"),
                stats.get("attackdamage"), stats.get("attackdamageperlevel"),
                stats.get("attackspeed"), stats.get("attackspeedperlevel"),
                stats.get("attackrange"), stats.get("movespeed"),
                stats.get("hpregen"), stats.get("hpregenperlevel"),
                stats.get("mpregen"), stats.get("mpregenperlevel"),
                stats.get("crit"), stats.get("critperlevel"),

                passive.get("name"),
                passive.get("description"),

                *q, *w, *e, *r,

                len(champ.get("skins", []))
            ]

            writer.writerow(row)

    print(f"✅ Patch {version} done")

csv_file.close()
print("\n🎉 CSV WITH SPELL EFFECTS COMPLETED")



=== PATCH 15.24.1 ===
✔ Found 172 champions
✅ Patch 15.24.1 done

🎉 CSV WITH SPELL EFFECTS COMPLETED


In [16]:
import pandas as pd

csv_path = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"

df = pd.read_csv(csv_path)

print(df.head())

     patch       id  key     name                  title        partype  \
0  15.24.1   Aatrox  266   Aatrox        Épée des Darkin  Puits de sang   
1  15.24.1     Ahri  103     Ahri  Renarde à neuf queues           Mana   
2  15.24.1    Akali   84    Akali       Assassin rebelle        Énergie   
3  15.24.1   Akshan  166   Akshan     Sentinelle rebelle           Mana   
4  15.24.1  Alistar   12  Alistar              Minotaure           Mana   

                tags  attack  defense  magic  ...    spell_e_cost  \
0            Fighter       8        4      3  ...               0   
1      Mage,Assassin       3        4      8  ...              60   
2           Assassin       5        3      8  ...              30   
3  Marksman,Assassin       0        0      0  ...              70   
4       Tank,Support       6        9      5  ...  50/55/60/65/70   

   spell_e_range                                    spell_e_effects  \
0          25000                                               

In [24]:
import requests
import tarfile
import io
import os
import json
import csv

# ================== CONFIG ==================
LANG = "fr_FR"

DEST_DIR = r"D:\lol draft analyzer\part 2\data champions\collect"
CSV_OUTPUT = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"
# ============================================

os.makedirs(DEST_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)

HEADERS = [
    "patch","id","key","name","title","partype","tags",
    "attack","defense","magic","difficulty",
    "hp","hpperlevel","mp","mpperlevel","armor","armorperlevel",
    "spellblock","spellblockperlevel",
    "attackdamage","attackdamageperlevel",
    "attackspeed","attackspeedperlevel",
    "attackrange","movespeed",
    "hpregen","hpregenperlevel",
    "mpregen","mpregenperlevel",
    "crit","critperlevel",
    "passive_name","passive_description",

    "spell_q_name","spell_q_maxrank","spell_q_cooldown","spell_q_cost","spell_q_range","spell_q_effects",
    "spell_w_name","spell_w_maxrank","spell_w_cooldown","spell_w_cost","spell_w_range","spell_w_effects",
    "spell_e_name","spell_e_maxrank","spell_e_cooldown","spell_e_cost","spell_e_range","spell_e_effects",
    "spell_r_name","spell_r_maxrank","spell_r_cooldown","spell_r_cost","spell_r_range","spell_r_effects",

    "skins_count"
]

# ================== CSV INIT ==================
def init_csv():
    if not os.path.exists(CSV_OUTPUT):
        with open(CSV_OUTPUT, "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow(HEADERS)

# ================== SPELL PARSER ==================
def extract_spell(spells, index):
    if len(spells) <= index:
        return [None]*6

    s = spells[index]
    effects = [e for e in s.get("effectBurn", []) if e and e != "0"]

    return [
        s.get("name"),
        s.get("maxrank"),
        s.get("cooldownBurn"),
        s.get("costBurn"),
        s.get("rangeBurn"),
        " | ".join(effects) if effects else None
    ]

# ================== CHAMP PARSER ==================
def parse_champion(champ, patch):
    info = champ.get("info", {})
    stats = champ.get("stats", {})
    passive = champ.get("passive", {})
    spells = champ.get("spells", [])

    q = extract_spell(spells, 0)
    w = extract_spell(spells, 1)
    e = extract_spell(spells, 2)
    r = extract_spell(spells, 3)

    return [
        patch,
        champ.get("id"),
        champ.get("key"),
        champ.get("name"),
        champ.get("title"),
        champ.get("partype"),
        ",".join(champ.get("tags", [])),

        info.get("attack"),
        info.get("defense"),
        info.get("magic"),
        info.get("difficulty"),

        stats.get("hp"), stats.get("hpperlevel"),
        stats.get("mp"), stats.get("mpperlevel"),
        stats.get("armor"), stats.get("armorperlevel"),
        stats.get("spellblock"), stats.get("spellblockperlevel"),
        stats.get("attackdamage"), stats.get("attackdamageperlevel"),
        stats.get("attackspeed"), stats.get("attackspeedperlevel"),
        stats.get("attackrange"), stats.get("movespeed"),
        stats.get("hpregen"), stats.get("hpregenperlevel"),
        stats.get("mpregen"), stats.get("mpregenperlevel"),
        stats.get("crit"), stats.get("critperlevel"),

        passive.get("name"),
        passive.get("description"),

        *q, *w, *e, *r,

        len(champ.get("skins", []))
    ]

# ================== PATCH PROCESSOR ==================
def process_patch(patch):
    print(f"\n=== PROCESSING PATCH {patch} ===")

    url = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{patch}.tgz"

    try:
        response = requests.get(url)
        response.raise_for_status()
    except Exception as e:
        print("❌ Download failed:", e)
        return

    tar_bytes = io.BytesIO(response.content)

    with tarfile.open(fileobj=tar_bytes, mode="r:gz") as tar:
        prefix = f"{patch}/data/{LANG}/champion/"

        members = [
            m for m in tar.getmembers()
            if m.name.startswith(prefix) and m.name.endswith(".json")
        ]

        print(f"✔ {len(members)} champions found")

        with open(CSV_OUTPUT, "a", newline="", encoding="utf-8") as csv_file:
            writer = csv.writer(csv_file)

            for m in members:
                try:
                    extracted = tar.extractfile(m)
                    raw = json.load(extracted)
                    champ = next(iter(raw["data"].values()))
                except Exception as e:
                    print("❌ JSON error:", m.name, e)
                    continue

                # Optional: save raw json
                out_json = os.path.join(DEST_DIR, os.path.basename(m.name))
                with open(out_json, "w", encoding="utf-8") as f:
                    json.dump(raw, f, ensure_ascii=False)

                row = parse_champion(champ, patch)
                writer.writerow(row)

    print(f"✅ Patch {patch} processed")

# ================== RUN ==================
if __name__ == "__main__":
    init_csv()

    PATCHES = [
        "15.24.1",
        "15.23.1",
        "15.22.1",
    ]

    for patch in PATCHES:
        process_patch(patch)

    print("\n🎉 ALL PATCHES DONE")



=== PROCESSING PATCH 15.24.1 ===
✔ 172 champions found
✅ Patch 15.24.1 processed

=== PROCESSING PATCH 15.23.1 ===
✔ 172 champions found
✅ Patch 15.23.1 processed

=== PROCESSING PATCH 15.22.1 ===
✔ 171 champions found
✅ Patch 15.22.1 processed

🎉 ALL PATCHES DONE


In [25]:
import pandas as pd

csv_path = r"D:\lol draft analyzer\part 2\data champions\champions_by_patch.csv"

df = pd.read_csv(csv_path)

print(df.head())

     patch       id  key     name                  title        partype  \
0  15.24.1   Aatrox  266   Aatrox        Épée des Darkin  Puits de sang   
1  15.24.1     Ahri  103     Ahri  Renarde à neuf queues           Mana   
2  15.24.1    Akali   84    Akali       Assassin rebelle        Énergie   
3  15.24.1   Akshan  166   Akshan     Sentinelle rebelle           Mana   
4  15.24.1  Alistar   12  Alistar              Minotaure           Mana   

                tags  attack  defense  magic  ...    spell_e_cost  \
0            Fighter       8        4      3  ...               0   
1      Mage,Assassin       3        4      8  ...              60   
2           Assassin       5        3      8  ...              30   
3  Marksman,Assassin       0        0      0  ...              70   
4       Tank,Support       6        9      5  ...  50/55/60/65/70   

   spell_e_range                                    spell_e_effects  \
0          25000                                               

In [27]:
print(df.columns)
print(df.info())
print(df.describe())

Index(['patch', 'id', 'key', 'name', 'title', 'partype', 'tags', 'attack',
       'defense', 'magic', 'difficulty', 'hp', 'hpperlevel', 'mp',
       'mpperlevel', 'armor', 'armorperlevel', 'spellblock',
       'spellblockperlevel', 'attackdamage', 'attackdamageperlevel',
       'attackspeed', 'attackspeedperlevel', 'attackrange', 'movespeed',
       'hpregen', 'hpregenperlevel', 'mpregen', 'mpregenperlevel', 'crit',
       'critperlevel', 'passive_name', 'passive_description', 'spell_q_name',
       'spell_q_maxrank', 'spell_q_cooldown', 'spell_q_cost', 'spell_q_range',
       'spell_q_effects', 'spell_w_name', 'spell_w_maxrank',
       'spell_w_cooldown', 'spell_w_cost', 'spell_w_range', 'spell_w_effects',
       'spell_e_name', 'spell_e_maxrank', 'spell_e_cooldown', 'spell_e_cost',
       'spell_e_range', 'spell_e_effects', 'spell_r_name', 'spell_r_maxrank',
       'spell_r_cooldown', 'spell_r_cost', 'spell_r_range', 'spell_r_effects',
       'skins_count'],
      dtype='object')
<cl

In [28]:
print(df["patch"].unique())



['15.24.1' '15.23.1' '15.22.1']


In [29]:
counts = df["id"].value_counts()

champions_less_than_3 = counts[counts < 3]

print(champions_less_than_3)

id
Zaahen    2
Name: count, dtype: int64


In [31]:
import requests
import tarfile
import io
import os
import json
import csv

# ================== CONFIG ==================
LANG = "fr_FR"

BASE_DIR = r"D:\lol draft analyzer\part 2\data champions"
TGZ_DIR = os.path.join(BASE_DIR, "tgz")
JSON_DIR = os.path.join(BASE_DIR, "collect")
CSV_DIR = os.path.join(BASE_DIR, "csv")

os.makedirs(TGZ_DIR, exist_ok=True)
os.makedirs(JSON_DIR, exist_ok=True)
os.makedirs(CSV_DIR, exist_ok=True)

# ================== UTILS ==================
def dump(value):
    """Serialize dict/list for CSV"""
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

# ================== SPELL FLATTENER ==================
def flatten_spell(spell, prefix):
    row = {}
    for k, v in spell.items():
        row[f"{prefix}_{k}"] = dump(v)
    return row

# ================== CHAMPION FLATTENER ==================
def flatten_champion(champ, patch):
    row = {
        "patch": patch,
        "id": champ.get("id"),
        "key": champ.get("key"),
        "name": champ.get("name"),
        "title": champ.get("title"),
        "partype": champ.get("partype"),
        "tags": dump(champ.get("tags")),
        "lore": champ.get("lore"),
        "blurb": champ.get("blurb"),
        "allytips": dump(champ.get("allytips")),
        "enemytips": dump(champ.get("enemytips")),
        "skins": dump(champ.get("skins")),
        "skins_count": len(champ.get("skins", [])),
    }

    # info
    for k, v in champ.get("info", {}).items():
        row[f"info_{k}"] = v

    # stats
    for k, v in champ.get("stats", {}).items():
        row[f"stats_{k}"] = v

    # passive
    passive = champ.get("passive", {})
    row["passive_name"] = passive.get("name")
    row["passive_description"] = passive.get("description")
    row["passive_image"] = dump(passive.get("image"))

    # spells
    spells = champ.get("spells", [])
    labels = ["q", "w", "e", "r"]

    for i, label in enumerate(labels):
        if i < len(spells):
            row.update(flatten_spell(spells[i], f"spell_{label}"))
        else:
            row[f"spell_{label}_missing"] = True

    return row

# ================== PATCH PROCESSOR ==================
def process_patch(patch):
    print(f"\n=== PATCH {patch} ===")

    tgz_path = os.path.join(TGZ_DIR, f"dragontail-{patch}.tgz")
    csv_path = os.path.join(CSV_DIR, f"champions_{patch}.csv")
    patch_json_dir = os.path.join(JSON_DIR, patch)
    os.makedirs(patch_json_dir, exist_ok=True)

    # ---- DOWNLOAD IF NEEDED ----
    if not os.path.exists(tgz_path):
        print("⬇ Downloading tgz...")
        url = f"https://ddragon.leagueoflegends.com/cdn/dragontail-{patch}.tgz"
        r = requests.get(url)
        r.raise_for_status()
        with open(tgz_path, "wb") as f:
            f.write(r.content)
    else:
        print("✔ Using cached tgz")

    # ---- OPEN ARCHIVE ----
    with tarfile.open(tgz_path, "r:gz") as tar:
        prefix = f"{patch}/data/{LANG}/champion/"
        members = [
            m for m in tar.getmembers()
            if m.name.startswith(prefix) and m.name.endswith(".json")
        ]

        print(f"✔ {len(members)} champions")

        rows = []

        for m in members:
            try:
                raw = json.load(tar.extractfile(m))
                champ = next(iter(raw["data"].values()))
            except Exception as e:
                print("❌ JSON error:", m.name, e)
                continue

            # save raw json
            with open(
                os.path.join(patch_json_dir, os.path.basename(m.name)),
                "w",
                encoding="utf-8"
            ) as f:
                json.dump(raw, f, ensure_ascii=False, indent=2)

            rows.append(flatten_champion(champ, patch))

    # ---- WRITE CSV ----
    if rows:
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

    print(f"✅ CSV written: {csv_path}")

# ================== RUN ==================
if __name__ == "__main__":
    PATCHES = [
        # "15.24.1",
        # "15.23.1",
        # "15.22.1",
        "15.21.1",
        "15.20.1",
        "15.19.1",
        "15.18.1",
        "15.17.1",
        "15.16.1",
        "15.15.1",
        "15.14.1",
        "15.13.1",
        "15.12.1",
        "15.11.1",
        "15.10.1",
        "15.9.1",
        "15.8.1",
        "15.7.1",
        "15.6.1",
        "15.5.1",
        "15.4.1",
        "15.3.1",
        "15.2.1",
        "15.1.1",
        "14.24.1",
    ]

    for patch in PATCHES:
        process_patch(patch)

    print("\n🎉 ALL PATCHES DONE")



=== PATCH 15.21.1 ===
⬇ Downloading tgz...
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.21.1.csv

=== PATCH 15.20.1 ===
⬇ Downloading tgz...
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.20.1.csv

=== PATCH 15.19.1 ===
⬇ Downloading tgz...
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.19.1.csv

=== PATCH 15.18.1 ===
⬇ Downloading tgz...
✔ 171 champions
✅ CSV written: D:\lol draft analyzer\part 2\data champions\csv\champions_15.18.1.csv

=== PATCH 15.17.1 ===
⬇ Downloading tgz...


KeyboardInterrupt: 

In [32]:
import os
import pandas as pd
import re

# ================== CONFIG ==================
CSV_DIR = r"D:\lol draft analyzer\part 2\data champions\csv"
OUTPUT_DIR = CSV_DIR
# ============================================

def extract_patch(filename):
    """
    Extrait le patch depuis champions_15.24.1.csv
    """
    match = re.search(r"champions_(\d+\.\d+\.\d+)\.csv", filename)
    return match.group(1) if match else None

def aggregate_csvs():
    csv_files = [
        f for f in os.listdir(CSV_DIR)
        if f.startswith("champions_") and f.endswith(".csv")
    ]

    if not csv_files:
        raise RuntimeError("❌ Aucun CSV trouvé")

    dfs = []
    patches = []

    for file in csv_files:
        patch = extract_patch(file)
        if not patch:
            print("⚠ Fichier ignoré:", file)
            continue

        path = os.path.join(CSV_DIR, file)
        print("✔ Loading:", file)

        df = pd.read_csv(path)
        dfs.append(df)
        patches.append(patch)

    # concat
    final_df = pd.concat(dfs, ignore_index=True)

    # tri par patch + champion
    final_df.sort_values(by=["patch", "id"], inplace=True)

    # nom du fichier final
    patches_sorted = sorted(
        patches,
        key=lambda p: tuple(map(int, p.split(".")))
    )

    first_patch = patches_sorted[0]
    last_patch = patches_sorted[-1]

    output_name = f"champions_{first_patch}_{last_patch}.csv"
    output_path = os.path.join(OUTPUT_DIR, output_name)

    final_df.to_csv(output_path, index=False, encoding="utf-8")

    print("\n🎉 AGGREGATION TERMINÉE")
    print("➡ Fichier généré :", output_path)
    print("➡ Lignes :", len(final_df))
    print("➡ Champions uniques :", final_df['id'].nunique())
    print("➡ Patchs :", final_df['patch'].nunique())

if __name__ == "__main__":
    aggregate_csvs()


✔ Loading: champions_15.18.1.csv
✔ Loading: champions_15.19.1.csv
✔ Loading: champions_15.20.1.csv
✔ Loading: champions_15.21.1.csv
✔ Loading: champions_15.22.1.csv
✔ Loading: champions_15.23.1.csv
✔ Loading: champions_15.24.1.csv

🎉 AGGREGATION TERMINÉE
➡ Fichier généré : D:\lol draft analyzer\part 2\data champions\csv\champions_15.18.1_15.24.1.csv
➡ Lignes : 1199
➡ Champions uniques : 172
➡ Patchs : 7


In [33]:
import pandas as pd

CSV_PATH = r"D:\lol draft analyzer\part 2\data champions\csv\champions_15.18.1_15.24.1.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)


Shape: (1199, 120)


In [34]:
print(df.head())
print(df.info())
print(df.describe())
print(df.describe(include="object"))
print(df.columns.tolist())


     patch       id  key     name                 title        partype  \
0  15.18.1   Aatrox  266   Aatrox       Épée des Darkin  Puits de sang   
1  15.18.1     Ahri  103     Ahri  Renard à neuf queues           Mana   
2  15.18.1    Akali   84    Akali      Assassin rebelle        Énergie   
3  15.18.1   Akshan  166   Akshan    Sentinelle rebelle           Mana   
4  15.18.1  Alistar   12  Alistar             Minotaure           Mana   

                       tags  \
0               ["Fighter"]   
1      ["Mage", "Assassin"]   
2              ["Assassin"]   
3  ["Marksman", "Assassin"]   
4       ["Tank", "Support"]   

                                                lore  \
0  Autrefois, Aatrox et ses frères étaient honoré...   
1  Connectée à la magie du royaume spirituel, Ahr...   
2  Ayant abandonné l'Ordre Kinkou et le titre de ...   
3  Se jouant du danger, Akshan combat le mal sans...   
4  Alistar est un guerrier redoutable cherchant à...   

                               

In [37]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

print(df.iloc[0])


patch                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [66]:
COLUMNS_TO_DROP = [
    "skins_count",
    "skins",
    "lore",
    "id",
    "title",
    "blurb",
    "allytips",
    "enemytips",
    "spell_q_description",
    "spell_w_description",
    "spell_e_description",
    "spell_r_description",
    "passive_description",
    "passive_name",
    "spell_q_name",
    "spell_w_name",
    "spell_e_name",
    "spell_r_name",
    "spell_r_tooltip",
    "spell_r_leveltip",
    "spell_e_tooltip",
    "spell_e_leveltip",
    "spell_w_tooltip",
    "spell_w_leveltip",
    "spell_q_tooltip",
    "spell_q_leveltip",
    "spell_q_image",
    "spell_w_image",
    "spell_e_image",
    "spell_r_image",
    "passive_image",

    "spell_q_cooldownBurn",
    "spell_q_costBurn",
    "spell_q_effectBurn",
    "spell_q_rangeBurn",
    "spell_w_cooldownBurn",
    "spell_w_costBurn",
    "spell_w_effectBurn",
    "spell_w_rangeBurn",
    "spell_e_cooldownBurn",
    "spell_e_costBurn",
    "spell_e_effectBurn",
    "spell_e_rangeBurn",
    "spell_r_cooldownBurn",
    "spell_r_costBurn",
    "spell_r_effectBurn",
    "spell_r_rangeBurn",
]

df_clean = df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [67]:
print(df_clean.iloc[0])

patch                                                                                                                                                                                                  15.18.1
key                                                                                                                                                                                                        266
name                                                                                                                                                                                                    Aatrox
partype                                                                                                                                                                                          Puits de sang
tags                                                                                                                                                                        

In [68]:
print(df_clean.shape)

(1199, 73)


In [69]:
OUTPUT_PATH = r"D:\lol draft analyzer\part 2\data champions\csv\champions_cleaned_15.18.1_15.24.1.csv"

df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("✅ CSV nettoyé sauvegardé :", OUTPUT_PATH)

✅ CSV nettoyé sauvegardé : D:\lol draft analyzer\part 2\data champions\csv\champions_cleaned_15.18.1_15.24.1.csv
